# Remote Sensing Image Segmentation with SatMAE++, SAM, and Wavelet-Based Strategies

**Authors**  
- Simone La Bella — Student ID: 1995847  
- Mattia Di Marco — Student ID: 2019367

## 1. Project Aim and Selected Papers

### Project Aim

The aim of this project is to investigate modern deep-learning strategies for **remote sensing image segmentation**, with particular focus on **building footprint segmentation** from aerial imagery.

The project uses the **INRIA Aerial Image Labeling Dataset** as the main downstream dataset and studies how different techniques can improve segmentation quality under both fully supervised and label-scarce conditions.

The main objectives are:

- evaluate the effectiveness of a **SatMAE++ pretrained Vision Transformer** as an encoder for building segmentation;
- compare the SatMAE++ based model with a **Swin-Unet baseline**;
- investigate whether **SAM-generated pseudo-labels** can reduce the amount of manually annotated data required for training;
- explore **wavelet-based strategies** for preserving spatial details and high-frequency information that may be useful for accurately segmenting building boundaries;
- perform controlled comparisons between the different approaches using common segmentation metrics such as **IoU, Dice score, precision, recall, and accuracy**.

### Selected Papers

The project is mainly inspired by the following papers.

#### 1. *Rethinking Transformers Pre-training for Multi-Spectral Satellite Imagery*

This work introduces **SatMAE++**, a self-supervised pretraining strategy designed specifically for remote sensing imagery. The method extends masked autoencoder pretraining to both RGB and multi-spectral satellite data and learns representations that can be transferred to downstream remote sensing tasks.

In this project, SatMAE++ provides the main pretrained encoder. The pretrained ViT-Large model is adapted from its original pretraining setting to a **dense prediction task**, namely binary building footprint segmentation on the INRIA dataset.

#### 2. *SAMRS: Scaling-up Remote Sensing Segmentation Dataset with Segment Anything Model*

SAMRS explores the use of the **Segment Anything Model (SAM)** to automatically generate segmentation masks for remote sensing imagery starting from spatial prompts such as bounding boxes.

Inspired by this approach, our project uses a **SatMAE segmentation teacher** to produce coarse building predictions. Connected components are converted into bounding-box prompts, which are then passed to SAM to generate refined pseudo-labels.

To avoid data leakage, the teacher is trained only on a small labeled subset of the INRIA training set. The remaining training images are treated as unlabeled data and are pseudo-labeled using the SatMAE teacher and SAM.

#### 3. *A Wavelet Diffusion GAN for Image Super-Resolution*

This paper investigates the integration of **wavelet transforms** into a generative super-resolution framework. Its main idea is to process image and feature representations in the wavelet domain so that low-frequency structural information and high-frequency details can be handled more effectively.

Instead of reproducing the diffusion-GAN architecture, we adapt the frequency-aware principle to the segmentation decoder.

#### 4. *WaveDM: Wavelet-Based Diffusion Models for Image Restoration*

WaveDM proposes a wavelet-based diffusion framework for image restoration. One of its central ideas is the separate treatment of **low-frequency and high-frequency components** in the wavelet domain.

This idea is adapted in our segmentation pipeline through a **wavelet-aware decoder**. Intermediate decoder features are decomposed using a Haar Discrete Wavelet Transform into low-frequency and high-frequency components.

The components are processed through separate branches, reconstructed using an inverse wavelet transform, and combined with the original features through a residual connection.


## 2. Theoretical Background and Key Concepts

### Semantic Segmentation

Semantic segmentation is a dense prediction task in which a class label is assigned to every pixel of an input image.

In this project, the problem is formulated as **binary semantic segmentation**:

- **building**;
- **background**.

Given an input aerial image $I$, the model produces a probability map

$$
P \in [0,1]^{H \times W}
$$

where each value represents the probability that the corresponding pixel belongs to a building.

A threshold, typically $0.5$, is then applied to obtain the final binary segmentation mask.

Building footprint segmentation is particularly challenging because the model must not only identify the presence of buildings, but also accurately reconstruct their **shape and boundaries**. Small buildings, adjacent structures, shadows, vegetation, and differences in illumination can make this task difficult.

---

### Vision Transformers for Remote Sensing

Vision Transformers (ViTs) process an image as a sequence of image patches rather than relying exclusively on local convolutional operations.

An image is divided into fixed-size patches, which are projected into embedding vectors and processed using Transformer blocks based on self-attention.

This allows the network to model long-range relationships between different regions of an image.

For remote sensing imagery, this property can be particularly useful because relevant objects and structures may cover large spatial regions and their interpretation can depend on the surrounding context.

In our project, the main encoder is based on a **ViT-Large architecture pretrained using SatMAE++**.

---

### Self-Supervised Pretraining and SatMAE++

Training large Vision Transformers from scratch generally requires large amounts of annotated data.

Self-supervised learning reduces this dependency by learning useful visual representations from unlabeled images.

SatMAE++ follows the idea of **Masked Autoencoders (MAE)**. During pretraining, a large fraction of the input image patches is masked and the model is trained to reconstruct the missing information from the visible patches.

Conceptually:

```text
Satellite image
      ↓
Split into patches
      ↓
Random masking
      ↓
Visible patches → Transformer encoder
      ↓
Reconstruction of masked information
```

Through this task, the encoder learns general representations of remote sensing imagery without requiring dense semantic labels.

The pretrained encoder can then be transferred to a downstream task through **fine-tuning**.

---

### Encoder-Decoder Segmentation Architecture

The SatMAE++ encoder produces a compact representation of the input image.

With an input resolution of $224 \times 224$ and patches of size $16 \times 16$, the spatial representation corresponds to a grid of:

$$
\frac{224}{16} \times \frac{224}{16}
=
14 \times 14
$$

The Transformer therefore produces high-level features at a significantly lower spatial resolution than the original image.

A segmentation decoder progressively increases the feature-map resolution:

```text
14 × 14
   ↓
28 × 28
   ↓
56 × 56
   ↓
112 × 112
   ↓
224 × 224
   ↓
Binary segmentation mask
```

The encoder is mainly responsible for extracting semantic information, while the decoder reconstructs the spatial resolution required for pixel-level prediction.

---

### Transfer Learning and Fine-Tuning

The pretrained SatMAE++ weights provide an initialization that already contains representations learned from remote sensing imagery.

During fine-tuning, both the encoder and segmentation decoder are optimized for the INRIA building segmentation task.

Different learning rates are used for the two components:

```text
Pretrained encoder → lower learning rate
New decoder        → higher learning rate
```

This allows the decoder to adapt quickly to the new task while modifying the pretrained representation more conservatively.

The project also allows comparison between pretrained and non-pretrained configurations to evaluate the actual contribution of SatMAE++ pretraining.

---

### Pseudo-Labeling and Semi-Supervised Learning

Dense segmentation masks are expensive to produce manually.

**Pseudo-labeling** is a semi-supervised learning technique in which a model trained on labeled data is used to generate artificial labels for unlabeled samples.

The newly labeled samples can then be added to the training set.

In our project, the INRIA training data is divided into:

```text
10% labeled data
90% treated as unlabeled data
```

A SatMAE++ teacher model is trained exclusively on the 10% labeled subset.

The remaining 90% is then processed by the teacher to obtain coarse building predictions.

This separation is important because the teacher must never use the ground-truth masks of the samples that will later receive pseudo-labels. Otherwise, the experiment would suffer from **data leakage**.

The leak-free pipeline is therefore:

```text
10% ground truth
      ↓
SatMAE teacher
      ↓
Predictions on unseen 90%
      ↓
Pseudo-label generation
      ↓
10% real labels + 90% pseudo-labels
      ↓
Final student model
```

---

### Segment Anything Model and Prompt-Based Segmentation

The **Segment Anything Model (SAM)** is a promptable segmentation model.

Instead of directly predicting a semantic class such as "building", SAM receives spatial prompts, for example:

- points;
- bounding boxes;
- masks.

and returns candidate segmentation masks corresponding to the specified region.

Because SAM is class-agnostic, it cannot directly identify which regions correspond to buildings.

For this reason, SatMAE is first used to produce a coarse semantic prediction.

Connected components are extracted from this prediction and converted into bounding boxes:

```text
INRIA image
     ↓
SatMAE teacher
     ↓
Coarse building mask
     ↓
Connected components
     ↓
Bounding-box prompts
     ↓
SAM
     ↓
Refined pseudo-label
```

SAM therefore acts as a **mask refinement mechanism**, while SatMAE provides the semantic information required to identify candidate building regions.

---

### Discrete Wavelet Transform

The Discrete Wavelet Transform (DWT) decomposes a signal into components representing different spatial-frequency information.

For a two-dimensional feature map, a one-level Haar DWT produces four components:

$$
LL,\quad LH,\quad HL,\quad HH
$$

where:

- **LL** represents the low-frequency component and mainly preserves coarse structures and global information;
- **LH**, **HL**, and **HH** represent high-frequency components associated with local variations, edges, and fine details.

Conceptually:

```text
Feature map
     ↓
    DWT
     ↓
┌────┬────┬────┬────┐
│ LL │ LH │ HL │ HH │
└────┴────┴────┴────┘
```

This decomposition is particularly relevant for building segmentation.

Low-frequency information can help represent the general shape and spatial extent of a building, while high-frequency information can contain useful information about building boundaries, corners, roof edges, separation between adjacent structures, and small spatial details.

---

### Wavelet-Aware Feature Processing

Inspired by wavelet-based image restoration and super-resolution approaches, the project introduces wavelet processing directly inside the segmentation decoder.

Instead of treating all feature information identically, intermediate feature maps are decomposed using the Haar DWT.

The low-frequency component and high-frequency components are then processed using separate branches:

```text
                     ┌→ LL → Low-Frequency Branch ─────┐
Feature map → DWT ───┤                                 │
                     └→ LH / HL / HH → High-Freq Branch
                                                       │
                                                       ↓
                                                      IDWT
                                                       ↓
                                              Reconstructed features
                                                       ↓
                                             Residual connection
```

The reconstructed wavelet features are combined with the original feature map through a residual connection.

This allows the network to preserve the original representation while learning additional frequency-aware corrections.

In the current architecture, wavelet-aware blocks are introduced at intermediate decoder resolutions:

```text
56 × 56
112 × 112
```

rather than at the initial $14 \times 14$ representation, where the spatial resolution is already highly compressed.

---

### Residual Learning

Residual connections combine the original representation $F$ with a learned transformation $G(F)$:

$$
F_{\mathrm{out}} = F + G(F)
$$

In the proposed wavelet-aware decoder, $G(F)$ corresponds to the features obtained after DWT, frequency-specific processing, and inverse DWT.

The residual formulation allows the model to retain the original decoder features while adding frequency-domain information only when it is useful.

---

### Training Objective

The main segmentation models are trained using a combination of **Binary Cross-Entropy (BCE)** and **Dice Loss**.

Binary Cross-Entropy evaluates pixel-wise classification:

$$
\mathcal{L}_{BCE}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\left[
y_i \log(p_i)
+
(1-y_i)\log(1-p_i)
\right]
$$

while Dice Loss directly encourages overlap between the predicted and ground-truth masks:

$$
\mathcal{L}_{Dice}
=
1 -
\frac{
2\sum_{i=1}^{N} p_i y_i + \epsilon
}{
\sum_{i=1}^{N} p_i +
\sum_{i=1}^{N} y_i +
\epsilon
}
$$

The combined objective used in the main experiments is:

$$
\mathcal{L}_{total}
=
0.5\,\mathcal{L}_{BCE}
+
0.5\,\mathcal{L}_{Dice}
$$

Using both objectives combines pixel-level supervision with an overlap-based objective particularly suitable for segmentation problems.